# Hospital Revenue — Forecasting Model

Notebook dự báo doanh thu và phân tích kịch bản:
1. **Data Loading** — Tải dữ liệu từ `hospital.db`
2. **Revenue Forecast** — Dự báo doanh thu 30 ngày tới (Holt-Winters)
3. **Scenario Analysis** — Phân tích kịch bản tăng/giảm theo %
4. **Export** — Xuất kết quả ra JSON cho dashboard

> Chạy xong Section 4, vào `http://127.0.0.1:5000/analysis` để xem kết quả.

## Section 1 — Data Loading

In [1]:
import sqlite3
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing

DB_PATH    = Path("../backend/hospital.db")
OUTPUT_DIR = Path("../backend/analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

FORECAST_DAYS = 30

print("Libraries loaded.")

Libraries loaded.


In [2]:
conn = sqlite3.connect(DB_PATH)

df_daily = pd.read_sql_query(
    "SELECT NGAY, THANHTIEN FROM FACT_DOANHTHU", conn
)
df_dept = pd.read_sql_query(
    """
    SELECT f.NGAY, f.THANHTIEN, k.TEN_KHOAPHONG
    FROM FACT_DOANHTHU f
    LEFT JOIN DIM_KHOAPHONG k ON f.ID_KHOAPHONG = k.ID_KHOAPHONG
    """,
    conn,
)
conn.close()

for d in [df_daily, df_dept]:
    d["NGAY"] = pd.to_datetime(d["NGAY"])

# Daily aggregation — fill missing dates with 0
daily = (
    df_daily.groupby("NGAY")["THANHTIEN"]
    .sum()
    .sort_index()
)
full_index = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_index, fill_value=0)

n_days = len(daily)
print(f"Khoảng thời gian: {daily.index.min().date()} → {daily.index.max().date()}")
print(f"Số ngày có dữ liệu: {n_days}")
daily.tail()

Khoảng thời gian: 2023-01-01 → 2026-05-11
Số ngày có dữ liệu: 1227


2026-05-07    217508533.0
2026-05-08    207235360.0
2026-05-09    153575654.0
2026-05-10    250206430.0
2026-05-11    295993797.0
Freq: D, Name: THANHTIEN, dtype: float64

## Section 2 — Revenue Forecast (30 ngày tới)

In [3]:
forecast_dates = pd.date_range(
    daily.index.max() + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D",
)

model_name = ""

if n_days >= 21:
    # Holt-Winters: trend + weekly seasonality
    try:
        model = ExponentialSmoothing(
            daily,
            trend="add",
            seasonal="add",
            seasonal_periods=7,
            initialization_method="estimated",
        )
        fit = model.fit(optimized=True)
        forecast = fit.forecast(FORECAST_DAYS)
        resid_std = float(fit.resid.std())
        steps = np.arange(1, FORECAST_DAYS + 1)
        lower = (forecast - 1.96 * resid_std * np.sqrt(steps)).clip(lower=0)
        upper =  forecast + 1.96 * resid_std * np.sqrt(steps)
        model_name = "Holt-Winters (trend + seasonal 7 ngày)"
        print(f"Model: {model_name}")
    except Exception as e:
        print(f"Holt-Winters thất bại ({e}), dùng naive.")
        n_days = 0  # trigger fallback

if n_days < 21:
    # Naive: lặp lại trung bình 7 ngày gần nhất
    base = daily.tail(7).mean()
    forecast = pd.Series([base] * FORECAST_DAYS, index=forecast_dates)
    lower    = forecast * 0.80
    upper    = forecast * 1.20
    model_name = "Naive (mean 7 ngày gần nhất)"
    print(f"Không đủ dữ liệu. Model: {model_name}")

forecast_monthly_total = float(forecast.sum())
last_30d_total         = float(daily.tail(30).sum())
growth_vs_last30       = (
    (forecast_monthly_total - last_30d_total) / last_30d_total * 100
    if last_30d_total > 0 else 0
)

print(f"\nDự báo tổng 30 ngày tới : {forecast_monthly_total:>15,.0f} VNĐ")
print(f"30 ngày thực tế trước đó: {last_30d_total:>15,.0f} VNĐ")
print(f"Tăng trưởng dự kiến      : {growth_vs_last30:>+14.1f}%")

Model: Holt-Winters (trend + seasonal 7 ngày)

Dự báo tổng 30 ngày tới :   6,623,535,850 VNĐ
30 ngày thực tế trước đó:   4,809,930,561 VNĐ
Tăng trưởng dự kiến      :          +37.7%


In [4]:

# Vẽ 60 ngày thực tế + 30 ngày dự báo
hist = daily.tail(60)

fig = go.Figure()

# Khoảng tin cậy 95%
fig.add_trace(go.Scatter(
    x=forecast_dates.strftime("%Y-%m-%d").tolist() + forecast_dates.strftime("%Y-%m-%d").tolist()[::-1],
    y=upper.tolist() + lower.tolist()[::-1],
    fill="toself",
    fillcolor="rgba(229, 57, 53, 0.12)",
    line=dict(color="rgba(0,0,0,0)"),
    name="Khoảng tin cậy 95%",
))

# Đường doanh thu thực tế
fig.add_trace(go.Scatter(
    x=hist.index.strftime("%Y-%m-%d").tolist(),
    y=hist.tolist(),
    name="Doanh thu thực tế",
    line=dict(color="#1976d2", width=2),
))

# Đường dự báo
fig.add_trace(go.Scatter(
    x=forecast_dates.strftime("%Y-%m-%d").tolist(),
    y=forecast.tolist(),
    name="Dự báo",
    line=dict(color="#e53935", width=2, dash="dash"),
))

# Đường phân cách (tách add_vline và add_annotation)
last_date = daily.index.max().strftime("%Y-%m-%d")
fig.add_vline(x=last_date, line_dash="dot", line_color="#78909c")
fig.add_annotation(
    x=last_date, y=1, yref="paper",
    text="Hôm nay", showarrow=False,
    xanchor="left", font=dict(color="#78909c", size=12),
)

fig.update_layout(
    title=f"Dự báo doanh thu 30 ngày tới — {model_name}",
    xaxis_title="Ngày",
    yaxis_title="Doanh thu (VNĐ)",
    hovermode="x unified",
)
fig.show()


## Section 3 — Scenario Analysis

In [5]:
# ── Kịch bản toàn viện ───────────────────────────────────────────────
SCENARIO_PCTS = [-20, -10, -5, 0, 5, 10, 20]

scenarios = [
    {
        "label"         : f"{pct:+.0f}%" if pct != 0 else "Base",
        "change_percent": pct,
        "monthly_total" : round(forecast_monthly_total * (1 + pct / 100)),
        "vs_last_30d"   : round(
            (forecast_monthly_total * (1 + pct / 100) - last_30d_total)
            / last_30d_total * 100
            if last_30d_total > 0 else 0,
            2,
        ),
    }
    for pct in SCENARIO_PCTS
]

# ── Kịch bản theo khoa ───────────────────────────────────────────────
dept_last30 = (
    df_dept[
        df_dept["NGAY"] > df_dept["NGAY"].max() - pd.Timedelta(days=30)
    ]
    .groupby("TEN_KHOAPHONG")["THANHTIEN"]
    .sum()
    .reset_index()
    .rename(columns={"THANHTIEN": "last_30d"})
)

# Base forecast phân bổ theo tỷ trọng khoa
total_last30 = dept_last30["last_30d"].sum()
dept_last30["weight"] = dept_last30["last_30d"] / total_last30 if total_last30 > 0 else 0
dept_last30["forecast_base"]   = (dept_last30["weight"] * forecast_monthly_total).round(0)
dept_last30["forecast_plus10"] = (dept_last30["forecast_base"] * 1.10).round(0)
dept_last30["forecast_plus20"] = (dept_last30["forecast_base"] * 1.20).round(0)
dept_last30["impact_plus10"]   = (dept_last30["forecast_base"] * 0.10).round(0)
dept_last30["impact_plus20"]   = (dept_last30["forecast_base"] * 0.20).round(0)

print("Scenario analysis hoàn tất.")
print(f"\nBảng kịch bản toàn viện:")
for s in scenarios:
    print(f"  {s['label']:>6} → {s['monthly_total']:>15,.0f} VNĐ  ({s['vs_last_30d']:+.1f}% vs 30 ngày qua)")

Scenario analysis hoàn tất.

Bảng kịch bản toàn viện:
    -20% →   5,298,828,680 VNĐ  (+10.2% vs 30 ngày qua)
    -10% →   5,961,182,265 VNĐ  (+23.9% vs 30 ngày qua)
     -5% →   6,292,359,058 VNĐ  (+30.8% vs 30 ngày qua)
    Base →   6,623,535,850 VNĐ  (+37.7% vs 30 ngày qua)
     +5% →   6,954,712,643 VNĐ  (+44.6% vs 30 ngày qua)
    +10% →   7,285,889,435 VNĐ  (+51.5% vs 30 ngày qua)
    +20% →   7,948,243,020 VNĐ  (+65.2% vs 30 ngày qua)


In [6]:
# Bar chart kịch bản toàn viện
colors = [
    "#e53935" if s["change_percent"] < 0
    else "#1976d2" if s["change_percent"] == 0
    else "#43a047"
    for s in scenarios
]

fig = go.Figure(go.Bar(
    x=[s["label"] for s in scenarios],
    y=[s["monthly_total"] for s in scenarios],
    marker_color=colors,
    text=[f"{s['monthly_total']:,.0f}" for s in scenarios],
    textposition="outside",
))
fig.update_layout(
    title="Scenario Analysis — Doanh thu dự báo 30 ngày theo kịch bản",
    yaxis_title="Doanh thu (VNĐ)",
    xaxis_title="Kịch bản",
    uniformtext_minsize=9,
)
fig.show()

In [7]:
# Grouped bar: base vs +10% vs +20% theo khoa
dept_sorted = dept_last30.sort_values("forecast_base", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=dept_sorted["TEN_KHOAPHONG"],
    x=dept_sorted["forecast_base"],
    name="Base", orientation="h", marker_color="#1976d2",
))
fig.add_trace(go.Bar(
    y=dept_sorted["TEN_KHOAPHONG"],
    x=dept_sorted["forecast_plus10"],
    name="+10%", orientation="h", marker_color="#43a047",
))
fig.add_trace(go.Bar(
    y=dept_sorted["TEN_KHOAPHONG"],
    name="+20%", orientation="h", marker_color="#26a69a",
    x=dept_sorted["forecast_plus20"],
))
fig.update_layout(
    title="Dự báo doanh thu theo khoa — 3 kịch bản",
    barmode="group",
    xaxis_title="Doanh thu (VNĐ)",
    height=500,
)
fig.show()

In [8]:
# Impact chart: nếu 1 khoa tăng 10%, tổng viện tăng bao nhiêu?
fig = px.bar(
    dept_sorted,
    x="impact_plus10", y="TEN_KHOAPHONG",
    orientation="h",
    title="Impact nếu mỗi khoa tăng 10% — doanh thu tăng thêm/tháng",
    labels={"impact_plus10": "Doanh thu tăng thêm (VNĐ)", "TEN_KHOAPHONG": ""},
    color="impact_plus10",
    color_continuous_scale="Greens",
    text_auto=",.0f",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Section 4 — Export JSON cho Dashboard

Chạy cell này để cập nhật trang `/analysis`.

In [9]:
generated_at = datetime.datetime.now().isoformat()

forecast_output = {
    "generated_at"      : generated_at,
    "model"             : model_name,
    "forecast_days"     : FORECAST_DAYS,
    "last_actual_date"  : daily.index.max().strftime("%Y-%m-%d"),
    "monthly_forecast"  : {
        "predicted_total"   : round(forecast_monthly_total),
        "lower_95"          : round(float(lower.sum())),
        "upper_95"          : round(float(upper.sum())),
        "growth_vs_last_30d": round(growth_vs_last30, 2),
    },
    "daily_forecast": [
        {
            "date"     : d.strftime("%Y-%m-%d"),
            "predicted": round(float(p)),
            "lower"    : round(float(lo)),
            "upper"    : round(float(up)),
        }
        for d, p, lo, up in zip(forecast_dates, forecast, lower, upper)
    ],
    "historical_last_60d": [
        {"date": d.strftime("%Y-%m-%d"), "actual": round(float(v))}
        for d, v in zip(hist.index, hist)
    ],
    "scenarios"         : scenarios,
    "by_department"     : dept_last30.fillna(0).round(0).to_dict(orient="records"),
}

out = OUTPUT_DIR / "forecast.json"
out.write_text(json.dumps(forecast_output, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"{out}")
print(f"\nExport hoàn tất lúc {generated_at[:19]}")
print(f"   Model     : {model_name}")
print(f"   Dự báo    : {forecast_monthly_total:,.0f} VNĐ ({growth_vs_last30:+.1f}% vs 30 ngày qua)")
print(f"   Dashboard : http://127.0.0.1:5000/analysis")

../backend/analysis_output/forecast.json

Export hoàn tất lúc 2026-06-23T01:57:16
   Model     : Holt-Winters (trend + seasonal 7 ngày)
   Dự báo    : 6,623,535,850 VNĐ (+37.7% vs 30 ngày qua)
   Dashboard : http://127.0.0.1:5000/analysis
